In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['pdf.fonttype'] = 42
%config InlineBackend.print_figure_kwargs={'facecolor': "w"}
%config InlineBackend.figure_format='retina'

os.makedirs("Figures", exist_ok=True)

In [ ]:
# all_obs.parquet can be downloaded from HF Hub via scripts/download.py
# See README.md for details
BASE_DIR = "/data/wholebrain_crispr_atlas"
METADATA_PATH = f"{BASE_DIR}/metadata/all_obs.parquet"

meta_full = pd.read_parquet(METADATA_PATH)
print(f"Total nuclei: {len(meta_full):,}")
print(f"Columns: {list(meta_full.columns)}")

In [ ]:
mask = (
    (meta_full["scDblFinder.class"] == "singlet") &
    (meta_full["num_genes"] >= 2000) &
    (meta_full["log_ambient_mse_norm"] > 0.09)
)
meta = meta_full[mask].copy()
print(f"After QC filter: {len(meta):,} cells")

## Region distribution

In [ ]:
neighborhood_frac = (
    meta["neighborhood"]
    .value_counts(normalize=True)
    .reset_index()
    .sort_values("proportion", ascending=False)
)
neighborhood_frac.columns = ["neighborhood", "fraction"]

plt.figure(figsize=(8, 8))
sns.barplot(data=neighborhood_frac, x="neighborhood", y="fraction",
            order=neighborhood_frac["neighborhood"], palette="tab10")
plt.ylabel("Fraction of cells")
plt.title("Neighborhood distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_Distribution_by_neighborhood.pdf")
plt.show()

In [ ]:
celltype_frac = (
    meta["predicted_group"]
    .value_counts(normalize=True)
    .reset_index()
    .sort_values("proportion", ascending=False)
)
celltype_frac.columns = ["celltype", "fraction"]

plt.figure(figsize=(8, 8))
sns.barplot(data=celltype_frac, x="celltype", y="fraction",
            order=celltype_frac["celltype"], palette="tab10")
plt.ylabel("Fraction of cells")
plt.title("Cell group distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_Distribution_by_celltype.pdf")
plt.show()

## Without unwanted regions (NA, MY, P)

In [ ]:
meta_sub = meta[~meta["region_level2"].isin(["NA", "MY", "P"])].copy()
meta_sub["region_level2"] = (
    meta_sub["region_level2"].astype("category").cat.remove_unused_categories()
)

In [ ]:
region_frac = (
    meta_sub["region_level2"]
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)
region_frac.columns = ["region", "fraction"]
region_frac = region_frac.sort_values("fraction", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=region_frac, x="region", y="fraction",
            order=region_frac["region"], palette="tab10")
plt.ylabel("Fraction of cells")
plt.title("Brain region distribution (excl. NA/MY/P)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_WB_Distribution_by_region_without_P_MY.pdf")
plt.show()

In [ ]:
region_frac = (
    meta["region_level1"]
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)
region_frac.columns = ["region", "fraction"]
region_frac = region_frac.sort_values("fraction", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=region_frac, x="region", y="fraction",
            order=region_frac["region"], palette="tab10")
plt.ylabel("Fraction of cells")
plt.title("Brain region distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_WB_Distribution_by_region_level1.pdf")
plt.show()

In [ ]:
region_frac = (
    meta["region_level2"]
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)
region_frac.columns = ["region", "fraction"]
region_frac = region_frac.sort_values("fraction", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=region_frac, x="region", y="fraction",
            order=region_frac["region"], palette="tab10")
plt.ylabel("Fraction of cells")
plt.title("Brain region distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_WB_Distribution_by_region_level2.pdf")
plt.show()

In [ ]:
meta["region_Yao"] = meta["region_level2"].copy()
meta["region_Yao"] = np.where(meta["region_Yao"] == "RHP",  "HPF", meta["region_Yao"])
meta["region_Yao"] = np.where(meta["region_Yao"] == "HIP",  "HPF", meta["region_Yao"])
meta["region_Yao"] = np.where(meta["region_Yao"].isin(["STRd", "STRv"]), "STR", meta["region_Yao"])
meta["region_Yao"] = np.where(meta["region_Yao"] == "sAMY", "STR", meta["region_Yao"])
meta["region_Yao"] = np.where(meta["region_Yao"] == "LSX",  "STR", meta["region_Yao"])

In [ ]:
region_palette = {
    "OLF": "#8C510A",
    "CTXsp": "#A6611A",
    "HPF": "#4DAC26",
    "Isocortex": "#1A9850",
    "STR": "#2C7BB6",
    "PAL": "#7B3294",
    "HY": "#D73027",
    "TH": "#F4A3A8",
    "MB": "#9E9E9E",
    "P": "#C2A5CF",
    "MY": "#C51B7D",
    "CB": "#FDBF11",
    "NA": "#D9D9D9"
}

In [ ]:
region_frac = (
    meta["region_Yao"]
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)
region_frac.columns = ["region", "fraction"]
region_frac = region_frac.sort_values("fraction", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=region_frac, x="region", y="fraction",
            order=region_frac["region"], palette="tab10")
plt.ylabel("Fraction of cells")
plt.title("Brain region distribution (Yao atlas)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_WB_Distribution_by_region_Yao.pdf")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=region_frac, x="region", y="fraction",
            order=region_frac["region"], palette=region_palette)
plt.ylabel("Fraction of cells")
plt.title("Brain region distribution (Yao atlas)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_WB_Distribution_by_region_Yao_palette.pdf")
plt.show()

## Cell type distribution

In [ ]:
cluster_frac = (
    meta["predicted_class"]
    .value_counts(normalize=True)
    .sort_index()
    .reset_index()
)
cluster_frac.columns = ["class", "fraction"]
cluster_frac = cluster_frac.sort_values("fraction", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=cluster_frac, x="class", y="fraction",
            order=cluster_frac["class"], palette="tab10")
plt.ylabel("Fraction of cells")
plt.title("Predicted class distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_WB_Distribution_by_predicted_class.pdf")
plt.show()

In [ ]:
class_palette = {
    "01 IT-ET Glut": "#E41A9C",
    "02 NP-CT-L6b Glut": "#F781BF",
    "03 OB-CR Glut": "#00BFC4",
    "04 DG-IMN Glut": "#00FFFF",
    "05 OB-IMN GABA": "#7F7F7F",
    "06 CTX-CGE GABA": "#B8DE29",
    "07 CTX-MGE GABA": "#A6D854",
    "08 CNU-MGE GABA": "#39FF14",
    "09 CNU-LGE GABA": "#4DAF4A",
    "10 LSX GABA": "#984EA3",
    "11 CNU-HYa GABA": "#6A3D9A",
    "12 HY GABA": "#377EB8",
    "13 CNU-HYa Glut": "#00CED1",
    "14 HY Glut": "#8A2BE2",
    "15 HY Gnrh1 Glut": "#FC8D62",
    "16 HY MM Glut": "#2ECC71",
    "17 MH-LH Glut": "#F1C40F",
    "18 TH Glut": "#1F78B4",
    "19 MB Glut": "#00441B",
    "20 MB GABA": "#FF69B4",
    "21 MB Dopa": "#7FFF00",
    "22 MB-HB Sero": "#F4A460",
    "23 P Glut": "#9B59B6",
    "24 MY Glut": "#6A51A3",
    "25 Pineal Glut": "#C51B7D",
    "26 P GABA": "#FF1493",
    "27 MY GABA": "#1E90FF",
    "28 CB GABA": "#00008B",
    "29 CB Glut": "#A0522D",
    "30 Astro-Epen": "#6B8E23",
    "31 OPC-Oligo": "#2C3E50",
    "32 OEC": "#4DBBD5",
    "33 Vascular": "#BDBDBD",
    "34 Immune": "#8C510A"
}

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=cluster_frac, x="class", y="fraction",
            order=cluster_frac["class"], palette=class_palette)
plt.ylabel("Fraction of cells")
plt.title("Predicted class distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("Figures/Fig1G_WB_Distribution_by_predicted_class_Yao.pdf")
plt.show()